# 5. Redes Neuronales Convolucionales (CNN)

En este notebook aprenderás a construir y entrenar redes convolucionales para clasificación de imágenes, usando TensorFlow/Keras. Se incluye teoría, código, visualizaciones, análisis de hiperparámetros y recomendaciones prácticas.

## Objetivo
- Comprender la teoría y arquitectura de las redes convolucionales (CNN).
- Implementar y entrenar una CNN para clasificación de imágenes.
- Analizar el impacto de los hiperparámetros y visualizar el aprendizaje.
- Visualizar filtros aprendidos para entender qué aprende la red.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado el [notebook 04 (MLP)](./04_redes_neuronales_capa_densa.ipynb).

- Conceptos de redes neuronales, funciones de activación y entrenamiento con Keras.

## 1. Introducción teórica

Una red neuronal convolucional (CNN) es un tipo de red diseñada para procesar datos con estructura de grilla, como imágenes.

- **Capas convolucionales:** Detectan patrones locales (bordes, texturas).
- **Capas de pooling:** Reducen la dimensionalidad y extraen características invariantes.
- **Capas densas:** Realizan la clasificación final.

**Ventajas:**
- Excelentes para imágenes y datos espaciales.
- Menos parámetros que una red densa equivalente (peso compartido).

**Desventajas:**
- Requieren más cómputo y datos que modelos clásicos.
- Arquitectura y ajuste de hiperparámetros más complejos.

## 2. Importación de librerías

In [ ]:
# === Reproducibilidad ===
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
tf.random.set_seed(SEED)

import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
%matplotlib inline

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 3. Carga y exploración del dataset

Usaremos MNIST (60,000 imágenes de 28x28 píxeles de dígitos escritos a mano).

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
print(f"Shape X_train: {X_train.shape}, y_train: {y_train.shape}")

fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f"{y_train[i]}")
    ax.axis('off')
plt.suptitle('Ejemplos de dígitos MNIST')
plt.show()

## 4. Preprocesamiento de datos

- Normalizaremos las imágenes a valores entre 0 y 1.
- Agregaremos la dimensión de canal (requerido por Conv2D).
- Convertiremos etiquetas a one-hot encoding.

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
X_train = np.expand_dims(X_train, -1)  # (60000, 28, 28, 1)
X_test = np.expand_dims(X_test, -1)

num_classes = len(np.unique(y_train))
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

print(f"Shape X_train: {X_train.shape}, y_train_cat: {y_train_cat.shape}")

## 5. Construcción y entrenamiento de la CNN

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_train, y_train_cat, epochs=10, batch_size=64,
                    validation_split=0.2, verbose=1)

## 6. Evaluación y visualización de resultados

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Accuracy (CNN): {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión - CNN')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()
print(classification_report(y_test, y_pred))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Entrenamiento')
axes[0].plot(history.history['val_accuracy'], label='Validación')
axes[0].set_title('Precisión durante el entrenamiento')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Entrenamiento')
axes[1].plot(history.history['val_loss'], label='Validación')
axes[1].set_title('Pérdida durante el entrenamiento')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### Visualización de filtros aprendidos

Visualizar los filtros de la primera capa convolucional nos muestra qué patrones detecta la red (bordes, texturas, etc.).

In [ ]:
# Obtener los pesos de la primera capa Conv2D
filters, biases = model.layers[0].get_weights()
print(f'Shape de filtros: {filters.shape}')  # (3, 3, 1, 32)

# Visualizar los primeros 16 filtros
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flatten()):
    if i < filters.shape[-1]:
        ax.imshow(filters[:, :, 0, i], cmap='gray')
    ax.axis('off')
plt.suptitle('Filtros aprendidos (capa Conv2D 1)')
plt.show()

## 7. Buenas prácticas: EarlyStopping y Dropout

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

model2 = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(num_classes, activation='softmax')
])

model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history2 = model2.fit(X_train, y_train_cat, epochs=20, batch_size=64,
                      validation_split=0.2, callbacks=[es], verbose=0)

best_epoch = np.argmax(history2.history['val_accuracy']) + 1
print(f"Mejor época: {best_epoch}")
print(f"Accuracy con EarlyStopping: {model2.evaluate(X_test, y_test_cat, verbose=0)[1]:.4f}")

### Recomendaciones prácticas para CNN

| Aspecto | Recomendación |
|---------|---------------|
| **Arquitectura** | Empieza simple, aumenta complejidad si es necesario |
| **Regularización** | Usa `Dropout` y/o `BatchNormalization` |
| **Filtros** | Más filtros en capas profundas (32 → 64 → 128) |
| **Early Stopping** | Monitorea `val_loss` con `patience=3-5` |
| **GPU** | Aprovecha la GPU para acelerar el entrenamiento |
| **Interpretación** | Visualiza filtros y activaciones para entender la red |

## 8. Discusión y Conclusiones

- Las CNN son el estándar para clasificación de imágenes.
- Dropout y EarlyStopping son esenciales para evitar sobreajuste.
- Visualizar filtros ayuda a interpretar qué aprende la red.
- En notebooks siguientes veremos RNN para secuencias y Transformers.

## 9. Ejercicios Propuestos

1. **Ejercicio 1:** Cambia el dataset a Fashion MNIST (`keras.datasets.fashion_mnist`). ¿Cómo cambia el accuracy?

2. **Ejercicio 2:** Agrega `BatchNormalization()` después de cada `Conv2D`. ¿Mejora la convergencia?

3. **Ejercicio 3:** Experimenta con Data Augmentation usando `keras.layers.RandomFlip` y `RandomRotation`. ¿Mejora la generalización?

4. **Ejercicio 4 (Avanzado):** Implementa una CNN con 3 bloques convolucionales (Conv2D + BN + MaxPool) y compárala con la arquitectura simple.

## 10. Referencias y Recursos

- [TensorFlow/Keras CNN](https://keras.io/examples/vision/mnist_convnet/)
- [CNN Explainer (Interactivo)](https://poloclub.github.io/cnn-explainer/)
- Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow.*

---

📎 **Notebook anterior:** [04. Redes Neuronales de Capa Densa](./04_redes_neuronales_capa_densa.ipynb)  
📎 **Notebook siguiente:** [06. Redes Recurrentes (RNN/LSTM)](./06_redes_recurrentes_rnn_lstm.ipynb)